In [1]:
# import sys
# from pathlib import Path

# # go one level up from notebooks/ to project root
# project_root = Path().resolve().parent

# sys.path.append(str(project_root))

In [2]:
import os
os.chdir("..")  # move from notebooks/ to project root

In [3]:
import pandas as pd
import numpy as np

In [4]:
from src.models.dataset import EvaluationDataset
from src.evaluation.multi_model_runner import run_multi_model_evaluation
from src.evaluation.judge import judge_correctness
from src.core.versioning import compute_dataset_version

In [5]:
dataset = EvaluationDataset.from_csv(path="data/dataset.csv")

models = ["mistral", "gemma2"]

In [6]:
dataset.samples

[QueryAnswerSample(query='What is a hedge?', expected_answer='An investment or contract intended to reduce the risk of adverse price movements in another asset. Common hedging instruments include futures, options, and currency forwards.'),
 QueryAnswerSample(query="What does 'underwriting' mean in insurance?", expected_answer='The process by which an insurer evaluates the risk of insuring an applicant, decides whether to offer coverage, and at what price and terms.'),
 QueryAnswerSample(query='What is 7 factorial (7!)?', expected_answer='5040'),
 QueryAnswerSample(query='Can my insurer cancel my policy after a claim?', expected_answer='Mid-term cancellation is usually limited to specific reasons (fraud, non-payment, material misrepresentation). At renewal, the insurer may decline to renew based on claims history, subject to local regulation.'),
 QueryAnswerSample(query='What is EBITDA?', expected_answer='Earnings Before Interest, Taxes, Depreciation, and Amortization. A proxy for opera

In [7]:
result_dataset = run_multi_model_evaluation(dataset, models)

[CACHE MISS] mistral
[CACHE MISS] gemma2
{"timestamp": "2026-05-09T08:47:33.504477", "level": "INFO", "message": "Starting inference with model=mistral", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-09T08:47:33.504903", "level": "INFO", "message": "Starting inference with model=gemma2", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-09T08:47:41.186373", "level": "INFO", "message": "Inference completed model=mistral", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-09T08:47:41.187446", "level": "INFO", "message": "Starting inference with model=llama3", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-09T08:47:42.107930", "level": "INFO", "message": "Inference completed model=llama3", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-09T08:47:42.109380", "level": "INFO", "message": "Starting inference with model=mistral", "module": "ollama_client", "requ

In [8]:
result_dataset.inferences["mistral"]

[InferenceResult(generated_answer=' In finance, a hedge refers to an investment or trading strategy that is intended to offset potential losses in another investment. The goal of hedging is to reduce the risk of adverse price movements in an asset, portfolio, or position.\n\nHedging strategies can be used for various purposes such as:\n\n1. Protecting against fluctuations in exchange rates (currency hedging)\n2. Guarding against changes in interest rates (interest rate hedging)\n3. Mitigating risks associated with commodity prices (commodity hedging)\n4. Reducing volatility due to market uncertainty or unforeseen events (risk management)\n\nFor example, an airline may enter into a currency hedge agreement to protect against fluctuations in the exchange rate between their local currency and the currencies they purchase fuel from. This helps them maintain a steady cost for their operations even if the exchange rate changes significantly during the contract period.', latency_seconds=7.682

In [9]:
result_dataset.inferences["gemma2"]

[InferenceResult(generated_answer='A hedge can have a few different meanings depending on the context:\n\n**1. Financial Hedge:** \nThis is the most common meaning. In finance, a hedge is an investment position taken to reduce the risk of adverse price movements in another asset. Imagine you own stock in a company that makes winter coats. If you think the price of wool might go up, making your coats more expensive to produce, you could buy futures contracts on wool. This would "hedge" your position against rising wool prices because if wool prices do go up, the value of your futures contract would also increase, offsetting some of your losses from the higher production costs.\n\n**2. Literal Hedge:** \nThis is a row of shrubs or trees planted close together to create a barrier for privacy or security.\n\n**3. Legal/Business Hedge:**  A hedge can also refer to a strategy used in business or law to minimize risk or liability. For example, a company might enter into contracts that protect

In [10]:
result_dataset.inferences["mistral"][0]

InferenceResult(generated_answer=' In finance, a hedge refers to an investment or trading strategy that is intended to offset potential losses in another investment. The goal of hedging is to reduce the risk of adverse price movements in an asset, portfolio, or position.\n\nHedging strategies can be used for various purposes such as:\n\n1. Protecting against fluctuations in exchange rates (currency hedging)\n2. Guarding against changes in interest rates (interest rate hedging)\n3. Mitigating risks associated with commodity prices (commodity hedging)\n4. Reducing volatility due to market uncertainty or unforeseen events (risk management)\n\nFor example, an airline may enter into a currency hedge agreement to protect against fluctuations in the exchange rate between their local currency and the currencies they purchase fuel from. This helps them maintain a steady cost for their operations even if the exchange rate changes significantly during the contract period.', latency_seconds=7.6829

In [11]:
result_dataset.inferences["mistral"][1]

InferenceResult(generated_answer=" In the context of insurance, underwriting refers to the process by which an insurance company evaluates the acceptance of an applicant and determines appropriate coverage, premiums, and risks for a particular insurance policy. The underwriting process involves assessing various factors such as the insured's age, health status, lifestyle, occupation, and other relevant details to evaluate the risk profile of the potential client. Based on this assessment, the insurance company will decide whether to issue the policy, decline it, or offer coverage with different terms and conditions, including premiums. The goal of underwriting is to ensure that the insurance company can accurately price and manage the risks associated with the policies they issue.", latency_seconds=5.2038798332214355, token_count=139, correctness=1.0)

In [12]:
models_preds = {
    "mistral": {
        "generated_answer": [],
        "latency_seconds": [],
        "token_count": [],
        "correctness": [],
    },
    "gemma2": {
        "generated_answer": [],
        "latency_seconds": [],
        "token_count": [],
        "correctness": [],
    },
}

for model in models:
    model_inference_results = result_dataset.inferences[model]
    for model_inference_result in model_inference_results:
        models_preds[model]["generated_answer"].append(model_inference_result.generated_answer)
        models_preds[model]["latency_seconds"].append(model_inference_result.latency_seconds)
        models_preds[model]["token_count"].append(model_inference_result.token_count)
        models_preds[model]["correctness"].append(model_inference_result.correctness)

In [47]:
models_preds_df = pd.DataFrame(models_preds)
mistral_preds_df = pd.DataFrame(models_preds["mistral"])
gemma_preds_df = pd.DataFrame(models_preds["gemma2"])

In [52]:
mistral_preds_df["correctness_over_latency"] = mistral_preds_df["correctness"] / mistral_preds_df["latency_seconds"]
gemma_preds_df["correctness_over_latency"] = gemma_preds_df["correctness"] / gemma_preds_df["latency_seconds"]

In [53]:
models_preds_df

,mistral,gemma2
generated_answer,"[ In finance, a hedge refers to an investment ...",[A hedge can have a few different meanings dep...
latency_seconds,"[7.682900905609131, 5.2038798332214355, 3.6114...","[13.747304201126099, 10.330981731414795, 1.835..."
token_count,"[199, 139, 96, 333, 317, 128, 178, 130, 143, 2...","[300, 222, 38, 380, 248, 18, 268, 330, 161, 34..."
correctness,"[0.9, 1.0, 1.0, 0.8, 0.9, 0.5, 0.75, 0.9, 0.9,...","[0.7, 0.9, 1.0, 0.8, 1.0, 0.5, 0.9, 0.9, 0.9, ..."


In [54]:
mistral_preds_df

,generated_answer,latency_seconds,token_count,correctness,correctness_over_latency
0,"In finance, a hedge refers to an investment o...",7.682901,199,0.9,0.117143
1,"In the context of insurance, underwriting ref...",5.203880,139,1.0,0.192164
2,Factorial of a number is the product of all p...,3.611438,96,1.0,0.276898
3,"Yes, an insurance company can potentially can...",12.971416,333,0.8,0.061674
4,"EBITDA stands for Earnings Before Interest, T...",11.995500,317,0.9,0.075028
...,...,...,...,...,...
296,"Tail coverage, in the context of software tes...",6.524011,161,0.0,0.000000
297,An insurance agent acts as a representative o...,9.965055,263,0.8,0.080281
298,It's important to notify your insurance compa...,5.140765,121,0.9,0.175071
299,A Python decorator is a special type of funct...,10.079153,264,0.9,0.089293


In [55]:
gemma_preds_df

,generated_answer,latency_seconds,token_count,correctness,correctness_over_latency
0,A hedge can have a few different meanings depe...,13.747304,300,0.7,0.050919
1,Underwriting in insurance is the process of **...,10.330982,222,0.9,0.087117
2,7! (7 factorial) is:\n\n7 * 6 * 5 * 4 * 3 * 2 ...,1.835027,38,1.0,0.544951
3,It depends on your specific insurance policy a...,17.416100,380,0.8,0.045935
4,"EBITDA stands for **Earnings Before Interest, ...",11.885605,248,1.0,0.084135
...,...,...,...,...,...
296,"""Tail coverage"" in the context of software tes...",9.377324,358,0.0,0.000000
297,An insurance agent acts as a **mediator** betw...,11.280446,432,0.7,0.062054
298,It's best to notify your insurance company **a...,5.556152,210,0.8,0.143985
299,A Python decorator is a powerful tool that all...,13.802711,529,0.8,0.057960


In [56]:
models_preds_df.to_csv("data/predictions/models_preds.csv", index=False)
mistral_preds_df.to_csv("data/predictions/mistral_preds.csv", index=False)
gemma_preds_df.to_csv("data/predictions/gemma_preds.csv", index=False)

In [57]:
mistral_preds_df[["latency_seconds", "token_count", "correctness", "correctness_over_latency"]].describe()

,latency_seconds,token_count,correctness,correctness_over_latency
count,301.000000,301.000000,301.000000,301.000000
mean,8.380791,213.435216,0.822625,0.153047
std,4.459801,114.327104,0.203626,0.180286
min,0.406551,8.000000,0.000000,0.000000
25%,5.154355,130.000000,0.800000,0.071788
50%,7.958181,199.000000,0.900000,0.104182
75%,10.677596,280.000000,0.900000,0.170830
max,23.853597,600.000000,1.000000,2.213744


In [58]:
gemma_preds_df[["latency_seconds", "token_count", "correctness", "correctness_over_latency"]].describe()

,latency_seconds,token_count,correctness,correctness_over_latency
count,301.000000,301.000000,301.000000,301.000000
mean,11.321711,286.059801,0.846811,0.184471
std,6.617986,148.124395,0.152606,0.308482
min,0.421489,10.000000,0.000000,0.000000
25%,6.526933,172.000000,0.800000,0.051139
50%,10.627947,320.000000,0.900000,0.078031
75%,16.117235,388.000000,0.900000,0.132624
max,28.429469,625.000000,1.000000,2.135288


In [59]:
mistral_preds_df.columns = [col + "_mistral" for col in mistral_preds_df.columns]
gemma_preds_df.columns = [col + "_gemma" for col in gemma_preds_df.columns]

all_preds_df = pd.concat([mistral_preds_df, gemma_preds_df], axis=1)

all_preds_df

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma
0,"In finance, a hedge refers to an investment o...",7.682901,199,0.9,0.117143,A hedge can have a few different meanings depe...,13.747304,300,0.7,0.050919
1,"In the context of insurance, underwriting ref...",5.203880,139,1.0,0.192164,Underwriting in insurance is the process of **...,10.330982,222,0.9,0.087117
2,Factorial of a number is the product of all p...,3.611438,96,1.0,0.276898,7! (7 factorial) is:\n\n7 * 6 * 5 * 4 * 3 * 2 ...,1.835027,38,1.0,0.544951
3,"Yes, an insurance company can potentially can...",12.971416,333,0.8,0.061674,It depends on your specific insurance policy a...,17.416100,380,0.8,0.045935
4,"EBITDA stands for Earnings Before Interest, T...",11.995500,317,0.9,0.075028,"EBITDA stands for **Earnings Before Interest, ...",11.885605,248,1.0,0.084135
...,...,...,...,...,...,...,...,...,...,...
296,"Tail coverage, in the context of software tes...",6.524011,161,0.0,0.000000,"""Tail coverage"" in the context of software tes...",9.377324,358,0.0,0.000000
297,An insurance agent acts as a representative o...,9.965055,263,0.8,0.080281,An insurance agent acts as a **mediator** betw...,11.280446,432,0.7,0.062054
298,It's important to notify your insurance compa...,5.140765,121,0.9,0.175071,It's best to notify your insurance company **a...,5.556152,210,0.8,0.143985
299,A Python decorator is a special type of funct...,10.079153,264,0.9,0.089293,A Python decorator is a powerful tool that all...,13.802711,529,0.8,0.057960


In [60]:
all_preds_df.loc[(all_preds_df["correctness_mistral"] > all_preds_df["correctness_gemma"])]

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma
0,"In finance, a hedge refers to an investment o...",7.682901,199,0.90,0.117143,A hedge can have a few different meanings depe...,13.747304,300,0.7,0.050919
1,"In the context of insurance, underwriting ref...",5.203880,139,1.00,0.192164,Underwriting in insurance is the process of **...,10.330982,222,0.9,0.087117
9,"In finance, Beta (β) is a measure of the risk...",9.996770,253,0.95,0.095031,"In finance, **beta (β)** measures a stock's vo...",16.337286,340,0.8,0.048968
16,To write a SQL query that counts the number o...,7.557624,174,0.80,0.105853,"```sql\nSELECT country, COUNT(*) AS customer_c...",7.441372,148,0.5,0.067192
30,Catastrophe (CAT) claims and regular insuranc...,19.212674,477,0.90,0.046844,Catastrophe (CAT) claims are distinct from reg...,18.971318,414,0.7,0.036898
...,...,...,...,...,...,...,...,...,...,...
280,A 'Claims-Made' policy is a type of insurance...,9.906242,249,1.00,0.100946,A **claims-made insurance policy** provides co...,9.395107,359,0.9,0.095795
294,"In Python, both `is` and `==` are used for co...",12.536989,317,0.90,0.071788,"In Python, both `is` and `==` are used for com...",9.071377,346,0.8,0.088189
297,An insurance agent acts as a representative o...,9.965055,263,0.80,0.080281,An insurance agent acts as a **mediator** betw...,11.280446,432,0.7,0.062054
298,It's important to notify your insurance compa...,5.140765,121,0.90,0.175071,It's best to notify your insurance company **a...,5.556152,210,0.8,0.143985


In [61]:
all_preds_df.loc[(all_preds_df["correctness_mistral"] < all_preds_df["correctness_gemma"])]

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma
4,"EBITDA stands for Earnings Before Interest, T...",11.995500,317,0.90,0.075028,"EBITDA stands for **Earnings Before Interest, ...",11.885605,248,1.0,0.084135
6,"In Python, you can convert a dictionary to a ...",7.172343,178,0.75,0.104568,You can use the `json` module in Python to ser...,12.556668,268,0.9,0.071675
10,Here's a simple Python context manager that t...,9.043648,226,0.20,0.022115,```python\nimport time\n\nclass Timer:\n de...,21.836662,464,0.8,0.036636
14,A depreciation holdback refers to a portion o...,8.154342,217,0.40,0.049054,A **depreciation holdback** is a provision in ...,18.899816,387,0.5,0.026455
15,The smallest unit of life that can independen...,5.088704,132,0.80,0.157211,The smallest unit of life is the **cell**. \n\...,5.339798,117,0.9,0.168546
...,...,...,...,...,...,...,...,...,...,...
286,The combined ratio is a measure used in the i...,8.088777,229,0.90,0.111265,The **Combined Ratio** is a key metric in insu...,8.744785,334,1.0,0.114354
288,"To add type hints to a Python function, you c...",14.654999,368,0.00,0.000000,Type hints in Python are annotations that spec...,9.759008,373,0.7,0.071729
289,To find all `.log` files modified within the ...,8.210119,216,0.80,0.097441,"```bash\nfind /var/log -type f -name ""*.log"" -...",4.574854,172,1.0,0.218586
292,You can count the number of lines in a file u...,6.525663,153,0.80,0.122593,```bash\nwc -l <filename> \n```\n\n**Explanati...,2.808803,103,0.9,0.320421


In [62]:
all_preds_df.loc[(all_preds_df["latency_seconds_mistral"] > all_preds_df["latency_seconds_gemma"])]

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma
2,Factorial of a number is the product of all p...,3.611438,96,1.0,0.276898,7! (7 factorial) is:\n\n7 * 6 * 5 * 4 * 3 * 2 ...,1.835027,38,1.0,0.544951
4,"EBITDA stands for Earnings Before Interest, T...",11.995500,317,0.9,0.075028,"EBITDA stands for **Earnings Before Interest, ...",11.885605,248,1.0,0.084135
5,The sum of angles in a triangle always adds u...,4.837385,128,0.5,0.103362,The sum of angles in a triangle is always **18...,0.925038,18,0.5,0.540518
16,To write a SQL query that counts the number o...,7.557624,174,0.8,0.105853,"```sql\nSELECT country, COUNT(*) AS customer_c...",7.441372,148,0.5,0.067192
18,"As of now, Mandarin Chinese is considered to ...",3.156082,80,0.9,0.285164,The language with the most native speakers is ...,1.790426,35,1.0,0.558526
...,...,...,...,...,...,...,...,...,...,...
289,To find all `.log` files modified within the ...,8.210119,216,0.8,0.097441,"```bash\nfind /var/log -type f -name ""*.log"" -...",4.574854,172,1.0,0.218586
290,The HTTP status code 401 indicates that the r...,4.993762,133,0.8,0.160200,HTTP status code **401 Unauthorized** means th...,2.987311,110,0.8,0.267799
292,You can count the number of lines in a file u...,6.525663,153,0.8,0.122593,```bash\nwc -l <filename> \n```\n\n**Explanati...,2.808803,103,0.9,0.320421
293,The boiling point of water at standard atmosp...,2.545785,65,0.9,0.353526,The boiling point of water at sea level is **1...,0.654164,19,1.0,1.528669


In [63]:
all_preds_df.loc[(all_preds_df["latency_seconds_mistral"] < all_preds_df["latency_seconds_gemma"])]

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma
0,"In finance, a hedge refers to an investment o...",7.682901,199,0.90,0.117143,A hedge can have a few different meanings depe...,13.747304,300,0.7,0.050919
1,"In the context of insurance, underwriting ref...",5.203880,139,1.00,0.192164,Underwriting in insurance is the process of **...,10.330982,222,0.9,0.087117
3,"Yes, an insurance company can potentially can...",12.971416,333,0.80,0.061674,It depends on your specific insurance policy a...,17.416100,380,0.8,0.045935
6,"In Python, you can convert a dictionary to a ...",7.172343,178,0.75,0.104568,You can use the `json` module in Python to ser...,12.556668,268,0.9,0.071675
7,"""Loss Adjustment Expense"" (LAE) refers to the...",5.023799,130,0.90,0.179147,Loss Adjustment Expense (LAE) refers to the **...,15.137478,330,0.9,0.059455
...,...,...,...,...,...,...,...,...,...,...
296,"Tail coverage, in the context of software tes...",6.524011,161,0.00,0.000000,"""Tail coverage"" in the context of software tes...",9.377324,358,0.0,0.000000
297,An insurance agent acts as a representative o...,9.965055,263,0.80,0.080281,An insurance agent acts as a **mediator** betw...,11.280446,432,0.7,0.062054
298,It's important to notify your insurance compa...,5.140765,121,0.90,0.175071,It's best to notify your insurance company **a...,5.556152,210,0.8,0.143985
299,A Python decorator is a special type of funct...,10.079153,264,0.90,0.089293,A Python decorator is a powerful tool that all...,13.802711,529,0.8,0.057960


In [65]:
all_preds_df.loc[(all_preds_df["correctness_over_latency_mistral"] > all_preds_df["correctness_over_latency_gemma"])]

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma
0,"In finance, a hedge refers to an investment o...",7.682901,199,0.90,0.117143,A hedge can have a few different meanings depe...,13.747304,300,0.7,0.050919
1,"In the context of insurance, underwriting ref...",5.203880,139,1.00,0.192164,Underwriting in insurance is the process of **...,10.330982,222,0.9,0.087117
3,"Yes, an insurance company can potentially can...",12.971416,333,0.80,0.061674,It depends on your specific insurance policy a...,17.416100,380,0.8,0.045935
6,"In Python, you can convert a dictionary to a ...",7.172343,178,0.75,0.104568,You can use the `json` module in Python to ser...,12.556668,268,0.9,0.071675
7,"""Loss Adjustment Expense"" (LAE) refers to the...",5.023799,130,0.90,0.179147,Loss Adjustment Expense (LAE) refers to the **...,15.137478,330,0.9,0.059455
...,...,...,...,...,...,...,...,...,...,...
295,"In an insurance policy, the deductible is the...",4.493146,119,0.90,0.200305,A **deductible** in an insurance policy is the...,7.331132,279,0.9,0.122764
297,An insurance agent acts as a representative o...,9.965055,263,0.80,0.080281,An insurance agent acts as a **mediator** betw...,11.280446,432,0.7,0.062054
298,It's important to notify your insurance compa...,5.140765,121,0.90,0.175071,It's best to notify your insurance company **a...,5.556152,210,0.8,0.143985
299,A Python decorator is a special type of funct...,10.079153,264,0.90,0.089293,A Python decorator is a powerful tool that all...,13.802711,529,0.8,0.057960


In [66]:
all_preds_df.loc[(all_preds_df["correctness_over_latency_mistral"] < all_preds_df["correctness_over_latency_gemma"])]

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma
2,Factorial of a number is the product of all p...,3.611438,96,1.0,0.276898,7! (7 factorial) is:\n\n7 * 6 * 5 * 4 * 3 * 2 ...,1.835027,38,1.0,0.544951
4,"EBITDA stands for Earnings Before Interest, T...",11.995500,317,0.9,0.075028,"EBITDA stands for **Earnings Before Interest, ...",11.885605,248,1.0,0.084135
5,The sum of angles in a triangle always adds u...,4.837385,128,0.5,0.103362,The sum of angles in a triangle is always **18...,0.925038,18,0.5,0.540518
10,Here's a simple Python context manager that t...,9.043648,226,0.2,0.022115,```python\nimport time\n\nclass Timer:\n de...,21.836662,464,0.8,0.036636
15,The smallest unit of life that can independen...,5.088704,132,0.8,0.157211,The smallest unit of life is the **cell**. \n\...,5.339798,117,0.9,0.168546
...,...,...,...,...,...,...,...,...,...,...
289,To find all `.log` files modified within the ...,8.210119,216,0.8,0.097441,"```bash\nfind /var/log -type f -name ""*.log"" -...",4.574854,172,1.0,0.218586
290,The HTTP status code 401 indicates that the r...,4.993762,133,0.8,0.160200,HTTP status code **401 Unauthorized** means th...,2.987311,110,0.8,0.267799
292,You can count the number of lines in a file u...,6.525663,153,0.8,0.122593,```bash\nwc -l <filename> \n```\n\n**Explanati...,2.808803,103,0.9,0.320421
293,The boiling point of water at standard atmosp...,2.545785,65,0.9,0.353526,The boiling point of water at sea level is **1...,0.654164,19,1.0,1.528669


In [71]:
queries = []
expected_answers = []

for sample in dataset.samples:
    queries.append(sample.query)
    expected_answers.append(sample.expected_answer)

all_preds_df["query"] = queries
all_preds_df["expected_answer"] = expected_answers

In [72]:
all_preds_df.to_csv("all_preds_df.csv", index=False)